In [0]:
rawdf1=spark.read.csv("/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified", header=False, inferSchema=True).toDF("id","fname","lname","age","profession")
rawdf1.show(200,False)
display(rawdf1.take(20))
display(rawdf1.sample(.1))
rawdf1.printSchema()
rawdf1.count()


In [0]:
print(rawdf1.columns)
print(rawdf1.dtypes)
for i in rawdf1.dtypes:
    if i[1]=='string':
        print(i[0])
print(rawdf1.schema)

print("actual count",rawdf1.count())
print("distinct count",rawdf1.distinct().count())
print("distinct count",rawdf1.select("fname").distinct().count(")


In [0]:
print("actual count",rawdf1.count())
print("de-duplicated record all columns count",rawdf1.distinct().count())
print("de-duplicated record all columns count",rawdf1.dropDuplicates().count())
print("de-duplicated record given col",rawdf1.dropDuplicates(['id']).count())
display(rawdf1.describe())
display(rawdf1.summary())

In [0]:
from pyspark.sql import SparkSession
spark= SparkSession.builder.appName("BB2-ETL Pipeline").getOrCreate()

In [0]:
#single file
struct1="cusid int, first_name string, last_name string, age int, profession string"
rawdf2=spark.read.schema(struct1).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified").show(truncate=False)
#multiple file
struct1="cusid int, first_name string, last_name string, age int, profession string"
rawdf2=spark.read.schema(struct1).csv(path=["/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified","/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified"]).show(truncate=False)



In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
struct_type=StructType([StructField("cusid", IntegerType(),False), StructField("first_name", StringType(),False),StructField("last_name", StringType(),False), StructField("age", IntegerType(),False), StructField("profession", StringType(),False)])
rawdf2=spark.read.schema(struct_type).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one",recursiveFileLookup=True, pathGlobFilter="custsm*")
rawdf2.show(truncate=False)
rawdf2.printSchema()

### UnionByName 1st example

In [0]:
struct1="CUSID int, FIRST_NAME string, LAST_NAME string, AGE int, PROFESSION string"
raw_NY=spark.read.schema(struct1).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified_NY")
raw_NY.show(truncate=False)
struct2="CUSID int, FIRST_NAME string, AGE int, PROFESSION string, CITY string"
raw_TX=spark.read.schema(struct2).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified_TX")
raw_TX.show(truncate=False)
merged_NY_TX=raw_NY.unionByName(raw_TX,allowMissingColumns=True)
merged_NY_TX.show(truncate=False)


### Validation

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,ShortType
struct_type=StructType([StructField("cusid", IntegerType(),True), StructField("first_name", StringType(),True),StructField("last_name", StringType(),True), StructField("age", ShortType(),True), StructField("profession", StringType(),True),StructField("corrupt_record",StringType(),True)])
cleandf1=spark.read.schema(struct_type).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified",mode="PERMISSIVE") 
cleandf1.show(truncate=False)
cleandf1.count()

### Rejection strategy

In [0]:
from pyspark.sql.functions import col
rejecteddf=cleandf1.filter(col("corrupt_record").isNull())
rejecteddf.show(truncate=False)
print(len(rejecteddf.collect()))
rejecteddf=cleandf1.where("corrupt_record is not null")
print(len(rejecteddf.collect()))
display(rejecteddf)
#we can use both filter and where clause to filter out the corrupt records


### Cleansing using drop()

In [0]:
#df=df.na.drop(how="any") this function drops all the rows which has null values in any column, In spark every transformations prodices dataframe,
# transformations are applied on a Dataframe
str1="CUSID int, FIRST_NAME string, LAST_NAME string, AGE int, PROFESSION string"
cleanseddf1=spark.read.schema(str1).csv("/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified")
cleanseddf2=cleanseddf1.na.drop(how="any")
display(cleanseddf1.where("age is  not null"))
print(cleanseddf1.count())
display(cleanseddf2.where("age is not null"))
print(cleanseddf2.count())

In [0]:
cleanseddf3=cleanseddf1.na.drop(how="any", subset=["cusid","age"])
display(cleanseddf3.where("cusid is not null"))
cleanseddf3=cleanseddf1.na.drop(how="all",subset=["cusid","age"])
display(cleanseddf3.where("cusid is not null"))

In [0]:
cleanseddf1=cleanseddf1.na.drop(how="all", subset=["first_name", "last_name"])
cleanseddf1.show()

### Scrubbing using fill() and replace()

In [0]:
scrubbeddf1=cleanseddf1.na.fill("not provided",subset=["last_name", "profession"])
scrubbeddf1.show(100,truncate=False)
replace_values1={"Actor":"Celebrity","Musician":"Composer"}
scrubbeddf2=scrubbeddf1.na.replace(replace_values1,subset=["profession"])
display(scrubbeddf2)


### Deduplication

In [0]:
#row-level
display(scrubbeddf2.where("cusid in ('4000001','4000003')"))
dedupdf1=scrubbeddf2.distinct() 
display(dedupdf1.coalesce(1).where("cusid in('4000001','4000003')")) 

#column-level
dedupdf2=dedupdf1.coalesce(1).dropDuplicates(subset=["cusid"])
display(dedupdf2.where("cusid = 4000003"))

#dedupdf2=dedupdf1.coalesce(1).where("cusid =4000003").orderBy(["cusid","age"],ascending=[True,False]).show()
dedupdf2=dedupdf1.coalesce(1).where("cusid =4000003").orderBy(["cusid","age"],ascending=[True,False]).dropDuplicates(subset=["cusid"])
display(dedupdf2.where("cusid = 4000003"))

### Standardization

#####Standardization1 - Column Enrichment (Addition of columns)

In [0]:
from pyspark.sql.functions import lit,initcap,col
standarddf1=scrubbeddf2.withColumn("sourcesystem",lit("retail"))
display(standarddf1.limit(20))

####Standardization2 - Column Uniformity

### withColumn

In [0]:
from pyspark.sql.functions import upper
display(dedupdf1.groupby("profession").count())
standarddf2=dedupdf1.withColumn("profession",initcap(col("profession")))
display(standarddf2)
standarddf3=standarddf2.withColumn("source system", lit("Retail"))
display(standarddf3)

###  Standardization3 - Format Standardization

In [0]:
from pyspark.sql.functions import lit,col,initcap,upper
standarddf3=rawdf1.where("id rlike '[a-z-|$]' ")
standarddf3.show()   #this is regualr expression like function
standarddf3=rawdf1.where("age  rlike '[^0-9]'")
standarddf3.show()
standarddf4=rawdf1.withColumn("Source System",lit("Retail")) #I did column enrichment here by adding a column since I used a different dataframe for column enrichment earlier
standarddf4.show(truncate=False)


In [0]:
from pyspark.sql.functions import *
replace_val={"one":'1',"two":'2',"three":'3',"four":'4',"five":'5',"six":'6',"seven":'7',"eight":'8',"nine":'9',"ten":'10'}
standarddf5= standarddf4.na.replace(replace_val,subset=["id"]) #applying to the values, completely changes it
standarddf6=standarddf5.withColumn("age",regexp_replace("age","-",""))#pattern based
display(standarddf6)

In [0]:
standarddf6.printSchema()
standarddf7=standarddf6.withColumns("id",)